# STAT5605 Project — Analysis Figures

This notebook generates all publication-quality figures for the project.  
**Data root:** `../data/`  |  **Output root:** `../outputs/figures/`

## 0. Imports & Global Style

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm, ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ── Output directory ──────────────────────────────────────────────────────────
FIG_OUT = '../outputs/figures'
os.makedirs(FIG_OUT, exist_ok=True)

# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.facecolor':  '#FAFAF8',
    'axes.facecolor':    '#F3F0EB',
    'axes.edgecolor':    '#AAAAAA',
    'axes.labelcolor':   '#2A2A2A',
    'xtick.color':       '#555555',
    'ytick.color':       '#555555',
    'text.color':        '#1A1A1A',
    'grid.color':        '#DDDDDD',
    'grid.linewidth':    0.6,
})

BG        = '#FAFAF8'
PANEL_BG  = '#F3F0EB'

# Injury-class colour palette (A / B / C)
INJ_COLORS = {
    'A': '#C0392B',   # deep crimson  — Suspected Serious
    'B': '#E67E22',   # amber         — Suspected Minor
    'C': '#2980B9',   # steel blue    — Possible
}
INJ_LABELS = {
    'A': 'Suspected Serious (A)',
    'B': 'Suspected Minor (B)',
    'C': 'Possible Injury (C)',
}

# Precipitation category definitions (mm/hr)
PRECIP_BINS   = [0.0, 0.1, 2.5, 7.6, 25.0, np.inf]
PRECIP_LABELS = ['Dry', 'Light Rain', 'Moderate Rain', 'Heavy Rain', 'Extreme Rain']
PRECIP_THRESH = [
    r'$\leq 0.1$ mm/hr',
    r'$0.1 – 2.5$ mm/hr',
    r'$2.5 – 7.6$ mm/hr',
    r'$7.6 – 25$ mm/hr',
    r'$> 25$ mm/hr',
]

def save_fig(fig, name, dpi=220):
    """Save figure as both PNG and PDF."""
    for ext in ('png', 'pdf'):
        path = os.path.join(FIG_OUT, f'{name}.{ext}')
        fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    print(f'Saved → {FIG_OUT}/{name}.[png|pdf]')

print('Setup complete.')

## 1. Load Data

In [ ]:
# ── Crash + weather data ──────────────────────────────────────────────────────
crash = pd.read_csv('../data/clean_crash_data/crash_data_weather.csv')
print(f'crash_data_weather: {crash.shape[0]:,} rows × {crash.shape[1]} cols')
print(crash.dtypes)
crash.head(3)

In [ ]:
# ── Enhanced CT towns geodataframe (from data_cleaning.ipynb) ─────────────────
ct_towns = gpd.read_file('../data/ct_towns_enhanced.gpkg')
print(f'ct_towns: {ct_towns.shape[0]} towns, columns: {ct_towns.columns.tolist()}')

## 2. Derived Columns
Categorise precipitation intensity and verify injury class values.

In [ ]:
# ── Confirm precipitation column name ────────────────────────────────────────
# Adjust 'precip_intensity_mm_hr' below to match the actual column in your CSV
PRECIP_COL  = 'precip_intensity_mm_hr'   # <-- update if needed
INJURY_COL  = 'Most Severe Injury'

crash['precip_cat'] = pd.cut(
    crash[PRECIP_COL],
    bins=PRECIP_BINS,
    labels=PRECIP_LABELS,
    right=True,
)

# Keep only crashes with valid injury class A / B / C
valid_injury = crash[INJURY_COL].isin(['A', 'B', 'C'])
print(f'Rows with valid injury class: {valid_injury.sum():,} / {len(crash):,}')
crash_v = crash[valid_injury].copy()

print('\nInjury class counts:')
print(crash_v[INJURY_COL].value_counts())

print('\nPrecip category counts:')
print(crash_v['precip_cat'].value_counts().sort_index())

---
## Figure 1 — Injury Severity by Precipitation Intensity (1 × 4 Pie Charts)

In [ ]:
# ── Compute proportions from real data ───────────────────────────────────────
cats_present = [c for c in PRECIP_LABELS if c in crash_v['precip_cat'].cat.categories]
n_cats = len(cats_present)

# Build per-category proportion table
prop_table = (
    crash_v.groupby(['precip_cat', INJURY_COL], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['A', 'B', 'C'], fill_value=0)
)
prop_table_pct = prop_table.div(prop_table.sum(axis=1), axis=0)

# Share of all crashes per category
cat_share = crash_v['precip_cat'].value_counts(normalize=True).mul(100)

print('Injury proportions per precipitation category:')
print(prop_table_pct.round(3))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, n_cats,
    figsize=(4.5 * n_cats, 6.5),
    facecolor=BG,
)
if n_cats == 1:
    axes = [axes]

fig.suptitle(
    'Injury Severity Distribution by Precipitation Intensity',
    fontsize=18, fontweight='bold', color='#0D0D0D', y=0.97,
)
fig.text(
    0.5, 0.905,
    'Connecticut Motor Vehicle Crashes  ·  Injury Classes: Suspected Serious (A), Suspected Minor (B), Possible (C)',
    ha='center', fontsize=10, color='#666666', style='italic',
)
fig.add_artist(plt.Line2D([0.04, 0.96], [0.895, 0.895],
                          transform=fig.transFigure, color='#AAAAAA', lw=0.8))

pie_colors = [INJ_COLORS['A'], INJ_COLORS['B'], INJ_COLORS['C']]

for ax, cat in zip(axes, cats_present):
    ax.set_facecolor(PANEL_BG)
    vals = prop_table_pct.loc[cat, ['A', 'B', 'C']].values
    n_cat = prop_table.loc[cat].sum()
    share = cat_share.get(cat, 0.0)

    wedges, _, autotexts = ax.pie(
        vals,
        colors=pie_colors,
        autopct=lambda p: f'{p:.1f}%' if p > 4 else '',
        pctdistance=0.68,
        startangle=90,
        wedgeprops={'linewidth': 1.4, 'edgecolor': BG},
        radius=0.88,
    )
    # Explode serious-injury wedge slightly
    wedges[0].set_radius(0.92)

    for at in autotexts:
        at.set_fontsize(8.5)
        at.set_color('white')
        at.set_fontweight('bold')
        at.set_path_effects([pe.withStroke(linewidth=1.5, foreground='#00000055')])

    # Category label
    idx = PRECIP_LABELS.index(cat)
    ax.set_title(cat, fontsize=13, fontweight='bold', color='#1A1A1A', pad=10)
    ax.text(0, -1.08, PRECIP_THRESH[idx],
            ha='center', va='top', fontsize=9, color='#555555', style='italic',
            transform=ax.transData)
    ax.text(0, -1.26, f'{share:.2f}% of crashes  (n={n_cat:,})',
            ha='center', va='top', fontsize=8.5, color='#888888',
            transform=ax.transData)
    ax.set_aspect('equal')

# Shared legend
legend_patches = [
    mpatches.Patch(facecolor=INJ_COLORS[k], edgecolor='#BBBBBB',
                   linewidth=0.8, label=INJ_LABELS[k])
    for k in ['A', 'B', 'C']
]
leg = fig.legend(
    handles=legend_patches, loc='lower center', ncol=3,
    bbox_to_anchor=(0.5, 0.025),
    frameon=True, framealpha=0.95, edgecolor='#AAAAAA', facecolor=BG,
    fontsize=10, title='Most Severe Injury', title_fontsize=10.5,
    handlelength=1.6, handleheight=1.2, columnspacing=2.2,
    prop={'family': 'serif', 'size': 10},
)
leg.get_title().set_fontfamily('serif')
leg.get_title().set_fontweight('bold')

plt.subplots_adjust(top=0.84, bottom=0.18, wspace=0.10)
save_fig(fig, 'fig1_injury_by_precip')
plt.show()

---
## Figure 2 — Crash Count by Hour of Day and Precipitation Category

In [ ]:
HOUR_COL = 'Hour of the Day'   # <-- update if needed

hourly = (
    crash_v.groupby([HOUR_COL, 'precip_cat'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=cats_present, fill_value=0)
)

HOUR_COLORS = ['#457B9D', '#2A9D8F', '#E9C46A', '#E76F51', '#6A0572']

fig, ax = plt.subplots(figsize=(13, 5.5), facecolor=BG)
ax.set_facecolor(PANEL_BG)

for col, color in zip(cats_present, HOUR_COLORS):
    ax.plot(hourly.index, hourly[col], lw=2.2, color=color, label=col, marker='o',
            markersize=4, markerfacecolor='white', markeredgewidth=1.5)

ax.set_title('Crash Frequency by Hour of Day and Precipitation Intensity',
             fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('Hour of Day (0–23)', fontsize=11)
ax.set_ylabel('Number of Crashes', fontsize=11)
ax.set_xticks(range(0, 24))
ax.grid(axis='y', linestyle='--')
ax.legend(title='Precipitation Category', fontsize=9.5, title_fontsize=10,
          framealpha=0.9, edgecolor='#AAAAAA')
plt.tight_layout()
save_fig(fig, 'fig2_crashes_by_hour_precip')
plt.show()

---
## Figure 3 — Injury Severity by Route Class

In [ ]:
ROUTE_COL = 'Route Class'   # <-- update if needed

route_inj = (
    crash_v.groupby([ROUTE_COL, INJURY_COL])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['A', 'B', 'C'], fill_value=0)
)
route_inj_pct = route_inj.div(route_inj.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots(figsize=(12, 5.5), facecolor=BG)
ax.set_facecolor(PANEL_BG)

x = np.arange(len(route_inj_pct))
width = 0.28
for i, (inj, label) in enumerate(INJ_LABELS.items()):
    bars = ax.bar(x + (i - 1) * width, route_inj_pct[inj], width,
                  color=INJ_COLORS[inj], label=label, edgecolor='white', linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(route_inj_pct.index, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Share of Crashes (%)', fontsize=11)
ax.set_title('Injury Severity by Route Class', fontsize=15, fontweight='bold', pad=12)
ax.legend(title='Most Severe Injury', fontsize=9.5, title_fontsize=10,
          framealpha=0.9, edgecolor='#AAAAAA')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.grid(axis='y', linestyle='--')
plt.tight_layout()
save_fig(fig, 'fig3_injury_by_route_class')
plt.show()

---
## Figure 4 — Choropleth: Crash Rate per Road Mile by Town

In [ ]:
# Requires ct_towns_enhanced.gpkg and a town name column in crash data.
# Adjust TOWN_COL to match the town field in crash_data_weather.csv.
TOWN_COL = 'Town'   # <-- update if needed

crash_by_town = crash_v[TOWN_COL].value_counts().rename('crash_count').reset_index()
crash_by_town.columns = ['NAME20', 'crash_count']

towns_plot = ct_towns.merge(crash_by_town, on='NAME20', how='left')
towns_plot['crash_count'] = towns_plot['crash_count'].fillna(0)
towns_plot['crash_per_road_mi'] = towns_plot['crash_count'] / towns_plot['road_mi_total']

CT_CRS = 6433
towns_proj = towns_plot.to_crs(epsg=CT_CRS)

fig, ax = plt.subplots(1, 1, figsize=(14, 9), facecolor=BG)
ax.set_facecolor(BG)

towns_proj.plot(
    column='crash_per_road_mi',
    ax=ax,
    cmap='YlOrRd',
    edgecolor='#888888',
    linewidth=0.4,
    legend=True,
    legend_kwds={
        'label': 'Crashes per Road Mile',
        'orientation': 'horizontal',
        'fraction': 0.03, 'pad': 0.02,
        'shrink': 0.6,
    },
    missing_kwds={'color': '#DDDDDD', 'label': 'No data'},
)

ax.set_title('Crash Rate per Road Mile by Connecticut Town',
             fontsize=16, fontweight='bold', pad=14)
ax.set_axis_off()
plt.tight_layout()
save_fig(fig, 'fig4_choropleth_crash_rate')
plt.show()

---
## Figure 5 — Wind Speed vs. Injury Severity (Box Plots)

In [ ]:
# Assumes 'wind_speed_ms' column exists (computed from u10/v10 in data_cleaning)
# Adjust column name as needed.
WIND_COL = 'wind_speed_ms'   # <-- update if needed

groups = [crash_v.loc[crash_v[INJURY_COL] == cls, WIND_COL].dropna()
          for cls in ['A', 'B', 'C']]

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor=BG)
ax.set_facecolor(PANEL_BG)

bp = ax.boxplot(
    groups,
    patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 2},
    whiskerprops={'color': '#555555'},
    capprops={'color': '#555555'},
    flierprops={'marker': 'o', 'markersize': 3,
                'markerfacecolor': '#AAAAAA', 'alpha': 0.5},
    widths=0.5,
)
for patch, cls in zip(bp['boxes'], ['A', 'B', 'C']):
    patch.set_facecolor(INJ_COLORS[cls])
    patch.set_alpha(0.85)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels([INJ_LABELS[c] for c in ['A', 'B', 'C']], fontsize=10)
ax.set_ylabel('Wind Speed (m/s)', fontsize=11)
ax.set_title('Wind Speed at Time of Crash by Injury Severity',
             fontsize=14, fontweight='bold', pad=12)
ax.grid(axis='y', linestyle='--')
plt.tight_layout()
save_fig(fig, 'fig5_wind_by_injury')
plt.show()

---
## Figure 6 — Crash Count Heatmap: Day of Week × Hour of Day

In [ ]:
DOW_COL = 'Day of the Week'   # <-- update if needed

# Normalise day labels if numeric (0=Mon or 0=Sun depending on source)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heatmap_data = (
    crash_v.groupby([DOW_COL, HOUR_COL])
    .size()
    .unstack(fill_value=0)
)
# Reindex rows to canonical day order if string labels match
existing_days = [d for d in day_order if d in heatmap_data.index]
if existing_days:
    heatmap_data = heatmap_data.reindex(existing_days)

fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)

im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd',
               interpolation='nearest')
plt.colorbar(im, ax=ax, label='Number of Crashes', shrink=0.8)

ax.set_xticks(range(24))
ax.set_xticklabels(range(24), fontsize=8)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=9)
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_title('Crash Frequency Heatmap: Day of Week × Hour of Day',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
save_fig(fig, 'fig6_heatmap_dow_hour')
plt.show()